# Transformada Rápida de Fourier (FFT) y análisis espectral


## 1. Objetivos
- Construir señales discretas.
- Calcular la FFT y convertir índices en frecuencias físicas.
- Estimar amplitudes de un espectro unilateral.
- Analizar una señal de vibración con ruido y ventana Hann.

## 2. Herramientas y convenciones

Para una señal real muestreada a frecuencia `fs`, usaremos `np.fft.rfft`, que devuelve las componentes de frecuencia no negativas. `np.fft.rfftfreq` construye el eje de frecuencias.

La separación entre bins es:

$$\Delta f = \frac{f_s}{N} = \frac{1}{T_{\mathrm{obs}}}$$



Para una señal sinusoidal de amplitud conocida, alineada con un bin y sin ventana, la amplitud unilateral se estima como `2*abs(X)/N`, sin duplicar DC ni Nyquist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 3. Funciones auxiliares para visualizar una señal y su espectro

In [ ]:
def espectro_unilateral(x, fs, ventana=None):
    """Devuelve frecuencias y amplitudes aproximadas para una señal real.

    Si se utiliza una ventana, la normalización por suma(ventana)
    corrige aproximadamente la ganancia coherente de la ventana.
    """
    x = np.asarray(x, dtype=float)
    N = len(x)
    x_centrada = x - np.mean(x)

    if ventana is None:
        w = np.ones(N)
    else:
        w = np.asarray(ventana, dtype=float)
        if len(w) != N:
            raise ValueError("La ventana debe tener la misma longitud que x.")

    X = np.fft.rfft(x_centrada * w)
    f = np.fft.rfftfreq(N, d=1/fs)
    A = 2 * np.abs(X) / np.sum(w)

    # DC no se duplica; Nyquist tampoco cuando N es par.
    A[0] /= 2
    if N % 2 == 0:
        A[-1] /= 2
    return f, A


def graficar_senal_y_espectro(t, x, f, A, fmax=None, titulo="Análisis espectral"):
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))
    ax[0].plot(t, x, lw=0.9)
    ax[0].set_xlabel("Tiempo (s)")
    ax[0].set_ylabel("Amplitud")
    ax[0].set_title("Dominio temporal")
    ax[1].plot(f, A, lw=1)
    if fmax is not None:
        ax[1].set_xlim(0, fmax)
    ax[1].set_xlabel("Frecuencia (Hz)")
    ax[1].set_ylabel("Amplitud aproximada")
    ax[1].set_title("Dominio de la frecuencia")
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()

## 4. Ejemplo resuelto 1: señal compuesta de dos frecuencias

**Enunciado.** Analizar $x(t)=\sin(2\pi 5t)+0.5\sin(2\pi 12t)$ muestreada a 1000 Hz durante un segundo.

In [ ]:
fs = 1000
T = 1.0
N = int(fs * T)
t = np.arange(N) / fs

x = np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*12*t)

f, A = espectro_unilateral(x, fs)
graficar_senal_y_espectro(t, x, f, A, fmax=20,
                          titulo="Ejemplo 1: dos componentes sinusoidales")

print(f"Número de muestras: {N}")
print(f"Resolución espectral: {fs/N:.3f} Hz")

### Interpretación
Se esperan picos en 5 Hz y 12 Hz, con amplitudes aproximadas 1.0 y 0.5. Como las frecuencias coinciden con bins y se observa un número entero de ciclos, la amplitud se recupera con precisión numérica.

**Pregunta:** ¿qué ocurriría si solo se conservaran 0.5 segundos de datos? La separación entre bins pasaría a 2 Hz, por lo que la rejilla sería más gruesa.

In [ ]:
# Comparación usando solo los primeros 0.5 segundos
N_corto = int(0.5 * fs)
t_corto = t[:N_corto]
x_corto = x[:N_corto]
f_corto, A_corto = espectro_unilateral(x_corto, fs)

graficar_senal_y_espectro(t_corto, x_corto, f_corto, A_corto, fmax=20,
                          titulo="Misma señal observada durante 0.5 s")
print(f"Resolución espectral con 0.5 s: {fs/N_corto:.3f} Hz")

## 5. Ejemplo resuelto 2: vibración con ruido

**Enunciado.** Simular una medición con componentes de 30 Hz y 75 Hz, más ruido gaussiano. Usar una frecuencia de muestreo de 500 Hz durante 2 s.

In [ ]:
fs = 500
T = 2.0
N = int(fs*T)
t = np.arange(N)/fs
rng = np.random.default_rng(42)

x = 0.8*np.sin(2*np.pi*30*t)
x += 0.3*np.sin(2*np.pi*75*t)
x += 0.2*rng.normal(size=N)

f, A = espectro_unilateral(x, fs, ventana=np.hanning(N))
graficar_senal_y_espectro(t, x, f, A, fmax=120,
                          titulo="Ejemplo 2: vibración simulada con ruido")
print(f"Resolución espectral: {fs/N:.3f} Hz")

### Interpretación física
- Los picos cercanos a 30 Hz y 75 Hz corresponden a las componentes sinusoidales introducidas.
- El ruido eleva el fondo espectral.
- La ventana Hann reduce fuga espectral, pero ensancha los picos.
- Un pico espectral no prueba por sí solo que haya resonancia o una falla: se requiere contexto físico.

## 6. Comparación: sin ventana y con ventana Hann

In [ ]:
# Elegimos una frecuencia que no coincide exactamente con un bin
fs = 200
T = 1.0
N = int(fs*T)
t = np.arange(N)/fs
x = np.sin(2*np.pi*10.4*t)

f0, A0 = espectro_unilateral(x, fs)
f1, A1 = espectro_unilateral(x, fs, ventana=np.hanning(N))

plt.figure(figsize=(10, 4))
plt.plot(f0, A0, label="Sin ventana")
plt.plot(f1, A1, label="Ventana Hann")
plt.xlim(0, 30)
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Amplitud aproximada")
plt.title("Efecto de la ventana sobre la fuga espectral")
plt.legend()
plt.grid(True)
plt.show()

## 7. Ejercicio guiado: frecuencia de Nyquist y aliasing

La frecuencia de Nyquist es \(f_s/2\). Si una señal contiene frecuencias por encima de este valor, pueden aparecer como frecuencias aparentes más bajas.

Ejecuta el código y cambia la frecuencia de la señal y la frecuencia de muestreo.

In [ ]:
f_real = 70       # Hz
fs = 100          # Hz: Nyquist = 50 Hz
T = 1.0
N = int(fs*T)
t = np.arange(N)/fs
x = np.sin(2*np.pi*f_real*t)

f, A = espectro_unilateral(x, fs)
graficar_senal_y_espectro(t, x, f, A, fmax=fs/2,
                          titulo="Demostración de muestreo insuficiente")
print(f"Frecuencia real: {f_real} Hz")
print(f"Frecuencia de Nyquist: {fs/2} Hz")

## 8. Ejercicios para casa

### Ejercicio 1. Superposición y resolución
Analiza la señal
$$
x(t)=1.2\sin(2\pi 8t)+0.7\sin(2\pi 15t)
$$
con frecuencia de muestreo de 200 Hz y duración de 2 s.

1. Grafica la señal y su espectro.
2. Identifica las frecuencias dominantes y estima sus amplitudes.
3. Repite con solo 0.5 s de datos.
4. Explica cómo cambia la resolución.

### Ejercicio 2. Diagnóstico espectral
Simula
$$
x(t)=0.9\sin(2\pi 20t)+0.4\sin(2\pi 60t)+\eta(t),
$$
donde el ruido es gaussiano. Usa `fs = 400 Hz` y `T = 3 s`.

1. Grafica un fragmento temporal y el espectro.
2. Compara el espectro con y sin ventana Hann.
3. Estima las frecuencias dominantes.
4. Explica las limitaciones de interpretar un espectro con ruido.

**Entrega sugerida:** notebook con código, gráficas y conclusiones breves.